# **K-NN и SVM: объяснение через близость**

### Практика к урокам «K-NN» и «SVM» модуля «Нелинейные интерпретируемые модели»

В уроках мы сказали несколько вещей, которые стоит проверить на данных, а не принимать на веру:

1. объяснение прогноза K-NN — это сами соседи и правило их отбора: метрика и число $k$;
2. смена масштаба или метрики меняет соседей, а значит, и ответ на вопрос «почему»;
3. веса линейного SVM задают положение гиперплоскости, и читать их как важность признаков нельзя;
4. решение SVM зависит только от опорных векторов, и у ядрового SVM объяснение остается через них;
5. в высокой размерности ближайший и самый далекий сосед становятся почти неотличимы (Beyer et al., 1999).

Данные те же, что в уроке «K-NN»: `diabetes.csv` (Pima Indians Diabetes, 768 пациенток), признаки `Glucose`, `BMI` и `Age`, цель `Outcome` — есть ли диабет. Объясняемый объект — пациентка из строки 15 файла, объект $a$ из шага о масштабировании.

Все вычисления детерминированы: числа у вас совпадут с числами в заданиях тренажера. Приятной работы!

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.svm import SVC

RANDOM_STATE = 42
FEATURES = ['Glucose', 'BMI', 'Age']
PATIENT = 15                      # строка файла, считая с нуля: объект a из урока K-NN

df = pd.read_csv('https://github.com/SadSabrina/open-xai-materials/raw/refs/heads/main/data/diabetes.csv')
X, y = df[FEATURES], df['Outcome']
print(df.shape)
df.loc[[15, 18, 23], FEATURES + ['Outcome']]

## Данные

Три объекта из урока: $a$ (строка 15), $b$ (строка 18) и $c$ (строка 23).

## Часть 1. K-NN: объяснение — это соседи

Функция `neighbours` возвращает пять ближайших соседей объекта среди остальных 767 строк: номер строки, расстояние и `Outcome` соседа. Прогноз K-NN с $k=5$ — большинство голосов этих пяти, а доля соседей с `Outcome = 1` — его вероятность класса 1. Ниже соседи пациентки $a$ на исходных признаках, без масштабирования.

In [ ]:
def neighbours(Xs, i, k=5, p=2):
    """k ближайших соседей объекта i среди всех остальных строк Xs (метрика Минковского с параметром p)."""
    rest = Xs.drop(index=i)
    dist, idx = NearestNeighbors(n_neighbors=k, p=p).fit(rest).kneighbors(Xs.loc[[i]])
    return pd.DataFrame({'row': rest.index[idx[0]], 'distance': dist[0].round(3),
                         'Outcome': y.loc[rest.index[idx[0]]].values})

raw = neighbours(X, PATIENT)
raw

## Часть 2. SVM: опорные векторы

Делим выборку на обучающую и тестовую (25%, со стратификацией), стандартизуем признаки по обучающей части и обучаем линейный SVM с $C=1$. У `SVC` после обучения есть `support_` (номера опорных векторов в обучающей выборке), `n_support_` (их число по классам), `dual_coef_` (произведения $\alpha_i y_i$) и `intercept_` (свободный член $b$).

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
scaler = StandardScaler().fit(X_tr)
Z_tr, Z_te = scaler.transform(X_tr), scaler.transform(X_te)

svm = SVC(kernel='linear', C=1).fit(Z_tr, y_tr)
print('опорных векторов по классам:', svm.n_support_, ' всего:', svm.n_support_.sum(), 'из', len(Z_tr))
print('accuracy на тесте:', round(svm.score(Z_te, y_te), 3))
print('w =', dict(zip(FEATURES, svm.coef_[0].round(3))))

## Задания

1. **Соседи после стандартизации.** Стандартизуйте все три признака по всем 768 строкам ($z = (x-\mu)/\sigma$, `StandardScaler`) и найдите пять соседей пациентки $a$. Сколько из них с `Outcome = 1`? Что предскажет K-NN с $k = 5$ и сколько соседей совпало с соседями на исходных признаках?
2. **Смена метрики.** На стандартизованных признаках замените евклидову метрику ($p = 2$) на манхэттенскую ($p = 1$). Сколько из пяти соседей пациентки $a$ останется прежними?
3. **Масштаб и качество.** Сравните accuracy K-NN на тестовой части при $k = 1, 5, 15, 25$ на исходных и на стандартизованных признаках (масштаб — по обучающей части).
4. **Сколько объектов держит решение.** Сколько опорных векторов у линейного SVM с $C=1$ выше? Как меняется их число при $C = 0.01$, $0.1$ и $10$? Проверьте утверждение урока: обучите SVM заново только на опорных векторах и сравните $w$ и $b$.
5. **Веса SVM и масштаб.** Обучите тот же линейный SVM на исходных, нестандартизованных признаках и сравните `coef_` двух моделей. Совпадает ли порядок признаков по модулю веса?
6. **Ядровой SVM через опорные векторы.** Обучите SVM с RBF-ядром (`C=1`, `gamma='scale'`) на стандартизованных признаках из задания 1, на всех строках, кроме пациентки $a$. Разложите `decision_function` пациентки $a$ на вклады опорных векторов $\alpha_i y_i K(x_i, x)$ и проверьте, что их сумма плюс $b$ равна `decision_function`. Чьи вклады самые большие по модулю и какую долю от суммы модулей всех вкладов дают десять крупнейших по модулю вкладов?
7. **Проклятие размерности.** Допишите к трем стандартизованным признакам $d = 0, 10, 100, 1000$ шумовых признаков из $\mathcal{N}(0, 1)$ (`np.random.default_rng(0)`, один генератор на все $d$ по порядку) и посчитайте для пациентки $a$ отношение $D_{\max}/D_{\min}$ — расстояния до самого далекого и до ближайшего объекта. Что происходит с ним при росте $d$?

## Что стоит унести

Объяснение K-NN и SVM — это объекты, а не веса: соседи у K-NN и опорные векторы у SVM. Но оба объяснения принадлежат не только модели, а еще и выбору масштаба, метрики и ядра: тот же объект с теми же данными получает других соседей, а линейный SVM — другой порядок признаков по весу, если поменять масштаб. Поэтому вместе с объяснением называют и то, в каком пространстве и какой мерой считалась близость.

И у близости есть предел: с ростом числа признаков, не связанных с задачей, самый далекий объект подходит к ближайшему, и «сосед» перестает что-либо значить. Объяснять через близость можно, пока признаков немного и они осмысленны.